In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

df=pd.read_csv('../../data/IMDB Dataset.csv')

In [2]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
ps = PorterStemmer()
stop_words = set(stopwords.words('english'))


def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    words = text.split()

    words = [ps.stem(word) for word in words if word not in stop_words]

    return ' '.join(words)



In [4]:
# Apply preprocessing

df['clean_review'] = df['review'].apply(preprocess_text)

# Convert labels

df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

# TF-IDF

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['clean_review'])

y = df['sentiment']


In [5]:
# Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Logistic Regression

lr_model = LogisticRegression()

lr_model.fit(X_train, y_train)

lr_predictions = lr_model.predict(X_test)



In [6]:
# Evaluation

print("Accuracy:", accuracy_score(y_test, lr_predictions))
print("Precision:", precision_score(y_test, lr_predictions))
print("Recall:", recall_score(y_test, lr_predictions))

print(classification_report(y_test, lr_predictions))
print(confusion_matrix(y_test, lr_predictions))


Accuracy: 0.8864
Precision: 0.8765193903144897
Recall: 0.901567771383211
              precision    recall  f1-score   support

           0       0.90      0.87      0.88      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

[[4321  640]
 [ 496 4543]]


In [7]:
sample_review = "This movie was wonderful"
clean_review = preprocess_text(sample_review)
review_vector = vectorizer.transform([clean_review])
prediction = lr_model.predict(review_vector)

if prediction[0] == 1:
    print("Positive Review")
else:
    print("Negative Review")

Positive Review


In [12]:
sample_review = "it was bad"
clean_review = preprocess_text(sample_review)
review_vector = vectorizer.transform([clean_review])
prediction = lr_model.predict(review_vector)

if prediction[0] == 1:
    print("Positive Review")
else:
    print("Negative Review")

Negative Review
